# Lab 4: Spherical Harmonic Analysis

| | |
|---|---|
| **Module** | M4 — Spherical Harmonic Analysis |
| **Estimated time** | ~2.5 hours |
| **Prerequisites** | Lectures M4-1, M4-2; Homework 4, Problems 4.1–4.3 |
| **Textbook** | Blakely Ch. 6; lecture notes M4 |

---

## Learning Outcomes

By the end of this lab you will be able to:

1. **Implement** Legendre polynomials via the three-term recurrence relation
2. **Verify** orthogonality of associated Legendre functions numerically
3. **Synthesize** a global scalar field from a truncated set of spherical harmonic coefficients
4. **Compute** and interpret the Mauersberger-Lowes power spectrum

---

## How to use this notebook

- **`[PROVIDED]`** — run as-is
- **`[IMPLEMENT]`** — replace `raise NotImplementedError` with correct code
- **`[VALIDATE]`** — run to check; prints ✓ PASS or ✗ FAIL
- **`[EXPLORE]`** — starting point; modify freely

**Your response:** cells require written answers. Hints: `print(hints['key'])`.


## Background

Any function on the sphere satisfying Laplace's equation can be expanded in
**spherical harmonics** $Y_n^m(\theta,\phi)$.  Outside a source-free sphere of
radius $a$, the potential is

$$
V(r,\theta,\phi) = a \sum_{n=0}^{N}\sum_{m=-n}^{n}
  \left(\frac{a}{r}\right)^{n+1} g_n^m Y_n^m(\theta,\phi)
$$

where the real spherical harmonics are

$$
Y_n^m(\theta,\phi) = \begin{cases}
  \bar{P}_n^m(\cos\theta)\cos(m\phi) & m \geq 0 \\
  \bar{P}_n^{|m|}(\cos\theta)\sin(|m|\phi) & m < 0
\end{cases}
$$

and $\bar{P}_n^m$ are the **fully normalized** associated Legendre functions
(Schmidt semi-norm; Blakely §6.3):

$$
\bar{P}_n^m = \sqrt{\frac{(2-\delta_{m0})(2n+1)(n-m)!}{(n+m)!}}\, P_n^m
$$

The $P_n^m$ can be computed stably via three-term recurrence (Blakely eq. 6.21–6.22),
which avoids the catastrophic cancellation that occurs in the direct factorial formula
for high degrees.

The **Mauersberger-Lowes spectrum** (or degree variance) is

$$
R_n = (n+1)\sum_{m=-n}^{n} \left(g_n^m\right)^2
$$

and describes how field power is distributed across spatial scales.
Its slope on a semi-log plot reveals whether the field is dominated by
deep sources (shallow slope at low $n$) or shallow sources (steep slope at high $n$).


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import lpmv, factorial

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})
print('Setup complete.')

In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
hints = {
    'p1_recurrence': (
        "The three-term recurrence for Legendre polynomials P_n(x):\n"
        "  P_0(x) = 1\n"
        "  P_1(x) = x\n"
        "  P_n(x) = [(2n-1)*x*P_{n-1}(x) - (n-1)*P_{n-2}(x)] / n   for n >= 2\n"
        "Build up an array from n=0 to n=N_max."
    ),
    'p1_orthogonality': (
        "Orthogonality: integral_{-1}^{1} P_n(x)*P_m(x) dx = 0 if n!=m,\n"
        "             = 2/(2n+1) if n==m.\n"
        "Use np.trapz (or scipy.integrate.quad) over x = cos(theta), dx = d(cos(theta))."
    ),
    'p2_assoc_legendre': (
        "Associated Legendre functions P_n^m(x) for m >= 0 can be computed from\n"
        "P_n via: P_n^m = (-1)^m * (1-x^2)^(m/2) * d^m P_n / dx^m\n"
        "But in practice use scipy.special.lpmv(m, n, x) for the un-normalized version,\n"
        "then apply the Schmidt normalization factor manually."
    ),
    'p2_normalization': (
        "Schmidt semi-norm factor for m > 0:\n"
        "  N_n^m = sqrt( 2*(2n+1)*factorial(n-m) / factorial(n+m) )\n"
        "For m = 0:\n"
        "  N_n^0 = sqrt( (2n+1) )\n"
        "Then Pbar_n^m = N_n^m * P_n^m(x)."
    ),
    'p3_synthesis': (
        "Loop over all (n, m) pairs in your coefficient table.\n"
        "For each grid point (theta_i, phi_j):\n"
        "  - Compute Pbar_n^m(cos(theta_i))\n"
        "  - Multiply by cos(m*phi_j) [for g_nm] or sin(|m|*phi_j) [for h_nm]\n"
        "  - Accumulate into V(theta_i, phi_j)\n"
        "Vectorize over the grid to avoid a Python loop over grid points."
    ),
    'p3_spectrum': (
        "The degree variance is R_n = (n+1) * sum_{m=-n}^{n} (coeff)^2.\n"
        "For real harmonics this means: R_n = (n+1) * [g_n0^2 + 2*sum_{m=1}^{n} (g_nm^2 + h_nm^2)]."
    ),
}
# print(hints['p1_recurrence'])

---
## Part 1: Legendre Polynomials *(Guided — ~40 min)*

The workhorse of spherical harmonic analysis is the Legendre polynomial.
You will implement them from the three-term recurrence — a numerically
stable approach that scales to high degree — then verify orthogonality.

**Available hints:** `p1_recurrence`, `p1_orthogonality`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Legendre polynomials via three-term recurrence.

def legendre_poly(N_max, x):
    """
    Compute Legendre polynomials P_0 through P_{N_max} at argument x.

    Parameters
    ----------
    N_max : int
        Maximum degree to compute.
    x : float or ndarray
        Argument; typically x = cos(theta).  Must satisfy -1 <= x <= 1.

    Returns
    -------
    P : ndarray, shape (N_max+1,) or (N_max+1, len(x))
        P[n] is P_n(x) for n = 0, 1, ..., N_max.

    Notes
    -----
    Recurrence (Blakely eq. 6.21):
        P_0 = 1,  P_1 = x
        P_n = [(2n-1)*x*P_{n-1} - (n-1)*P_{n-2}] / n,  n >= 2
    """
    x = np.asarray(x, dtype=float)
    P = np.zeros((N_max + 1,) + x.shape)

    # Step 1: seed values P_0 and P_1
    # YOUR CODE HERE
    raise NotImplementedError

    # Step 2: recurrence for n = 2, ..., N_max
    for n in range(2, N_max + 1):
        # YOUR CODE HERE
        raise NotImplementedError

    return P

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

def _check(label, got, expected, rtol=1e-6, atol=0.0):
    if np.allclose(got, expected, rtol=rtol, atol=atol):
        print(f'  ✓ PASS  {label}')
    else:
        print(f'  ✗ FAIL  {label}')
        print(f'    expected: {np.asarray(expected)}')
        print(f'    got:      {np.asarray(got)}')

from scipy.special import legendre as sp_legendre

_x = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
_P = legendre_poly(6, _x)

# Test against scipy for each degree
for _n in range(7):
    _expected = np.polyval(sp_legendre(_n), _x)
    _check(f'P_{_n}(x) matches scipy', _P[_n], _expected)

# P_n(1) = 1 for all n
_check('P_n(1) = 1 for all n', legendre_poly(10, np.array([1.0]))[:, 0], np.ones(11))

# P_n(-1) = (-1)^n
_check('P_n(-1) = (-1)^n', legendre_poly(5, np.array([-1.0]))[:, 0],
       np.array([1, -1, 1, -1, 1, -1], dtype=float))

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 1: Verify orthogonality numerically.

def legendre_inner_product(n, m, N_quad=1000):
    """
    Numerically compute the inner product <P_n, P_m> = integral_{-1}^{1} P_n(x)*P_m(x) dx.

    Parameters
    ----------
    n, m : int
        Degrees of the two polynomials.
    N_quad : int
        Number of quadrature points for the integral.

    Returns
    -------
    ip : float
        Numerical value of the inner product.

    Notes
    -----
    Analytically: <P_n, P_m> = 2/(2n+1) * delta_{nm}
    Use np.trapz over x = linspace(-1, 1, N_quad).
    """
    x = np.linspace(-1.0, 1.0, N_quad)
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────

# Off-diagonal should be ~0
for (_n, _m) in [(0, 1), (1, 2), (2, 4), (3, 5)]:
    _ip = legendre_inner_product(_n, _m)
    _check(f'<P_{_n}, P_{_m}> ≈ 0', _ip, 0.0, atol=1e-5)

# Diagonal should be 2/(2n+1)
for _n in range(6):
    _ip = legendre_inner_product(_n, _n)
    _check(f'<P_{_n}, P_{_n}> = 2/{2*_n+1}', _ip, 2.0/(2*_n+1), rtol=1e-4)

### Question 1.1 — Why use the recurrence relation?

You could also compute $P_n(x)$ directly from the Rodrigues formula or
the explicit polynomial coefficients.  Why is the three-term recurrence
preferred for geophysical applications where $n$ may reach 360 or higher?
What numerical issue arises with the direct approach?

**Your response:**

> *(Write 3–5 sentences.)*


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Plot P_0 through P_5 as a function of colatitude theta (0° to 180°).
# Requirements:
#   - x-axis: colatitude in degrees
#   - y-axis: P_n(cos theta)
#   - Each curve labeled with its degree n
#   - Mark the zeros of P_3 and P_5 with small vertical tick marks or points

theta_deg = np.linspace(0, 180, 500)
theta_rad = np.radians(theta_deg)
x = np.cos(theta_rad)

# YOUR CODE HERE

---
## Part 2: Associated Legendre Functions and Spherical Harmonics *(Supported — ~50 min)*

The associated Legendre functions $\bar{P}_n^m$ are the angular building blocks
of real spherical harmonics.  You will implement the Schmidt-normalized versions
and use them to visualize individual harmonics on the sphere.

**Available hints:** `p2_assoc_legendre`, `p2_normalization`


In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Schmidt-normalized associated Legendre function.
# No step-by-step skeleton — implement from the docstring.

def schmidt_Pnm(n, m, x):
    """
    Schmidt semi-normalized associated Legendre function Pbar_n^m(x).

    Parameters
    ----------
    n : int
        Degree (n >= 0).
    m : int
        Order (0 <= m <= n).
    x : float or ndarray
        Argument; x = cos(theta), -1 <= x <= 1.

    Returns
    -------
    Pbar : float or ndarray
        Schmidt-normalized P_n^m(x).

    Notes
    -----
    Normalization factor (Blakely eq. 6.18):
        N_n^m = sqrt(2*(2n+1)*factorial(n-m)/factorial(n+m))  for m > 0
        N_n^0 = sqrt(2n+1)                                     for m = 0
    Pbar_n^m = N_n^m * lpmv(m, n, x)
    where lpmv(m, n, x) is scipy.special.lpmv.
    Note: lpmv uses the Condon-Shortley phase convention (includes (-1)^m).
    To get the standard form without the CS phase, multiply lpmv by (-1)^m.
    """
    raise NotImplementedError

In [ ]:
# [VALIDATE] ──────────────────────────────────────────────────────────────────
# Schmidt semi-normalized orthogonality:
#   integral_{-1}^{1} Pbar_n^m * Pbar_n'^m dx = 2/(2n+1) * delta_{nn'}
# (no factor of 2 difference from the un-normalized case for m=0)

def _pnm_inner(n1, n2, m, N=2000):
    x = np.linspace(-1, 1, N)
    return np.trapz(schmidt_Pnm(n1, m, x) * schmidt_Pnm(n2, m, x), x)

# Zonal harmonics (m=0)
for _n in range(5):
    _check(f'<Pbar_{_n}^0, Pbar_{_n}^0> = 2/{2*_n+1}',
           _pnm_inner(_n, _n, 0), 2.0/(2*_n+1), rtol=1e-4)

# Off-diagonal
_check('<Pbar_2^1, Pbar_4^1> = 0', _pnm_inner(2, 4, 1), 0.0, atol=1e-5)
_check('<Pbar_3^2, Pbar_5^2> = 0', _pnm_inner(3, 5, 2), 0.0, atol=1e-5)

# Sectoral Pbar_n^n at x=0 (equator): non-zero for even n, zero for odd n via parity
# (just checking it evaluates without error)
_val = schmidt_Pnm(4, 4, 0.0)
if np.isfinite(_val):
    print(f'  ✓ PASS  Pbar_4^4(0) = {_val:.4f} (finite)')
else:
    print(f'  ✗ FAIL  Pbar_4^4(0) is not finite: {_val}')

In [ ]:
# [IMPLEMENT] ─────────────────────────────────────────────────────────────────
# Tier 2: Real spherical harmonic Y_n^m(theta, phi).

def real_sph_harm(n, m, theta, phi):
    """
    Real spherical harmonic Y_n^m(theta, phi).

    Parameters
    ----------
    n : int
        Degree.
    m : int
        Order (-n <= m <= n).  Positive m uses cosine; negative uses sine.
    theta : ndarray
        Colatitude, radians (0 at north pole).
    phi : ndarray
        Longitude, radians.

    Returns
    -------
    Y : ndarray
        Same shape as theta/phi.

    Notes
    -----
    Y_n^m  = Pbar_n^m(cos theta) * cos(m*phi)    for m >= 0
    Y_n^m  = Pbar_n^|m|(cos theta) * sin(|m|*phi) for m <  0
    """
    raise NotImplementedError

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Visualize a selection of real spherical harmonics as global maps.
#
# Make a 2×4 grid of panels showing Y_n^m for:
#   (n,m) = (1,0), (2,0), (2,1), (2,2), (3,0), (3,1), (4,0), (4,2)
#
# Use a diverging colormap (e.g. 'RdBu_r') centered at zero.
# x-axis: longitude (0–360°), y-axis: latitude (–90 to 90°).
# Title each panel with Y_{n}^{m}.

lon_deg = np.linspace(0, 360, 360)
lat_deg = np.linspace(-90, 90, 181)
LON, LAT = np.meshgrid(np.radians(lon_deg), np.radians(90 - lat_deg))  # phi, theta

harmonics = [(1,0), (2,0), (2,1), (2,2), (3,0), (3,1), (4,0), (4,2)]

# YOUR CODE HERE

### Question 2.1 — Nodal lines

A real spherical harmonic $Y_n^m$ has $(n - m)$ **nodal latitude lines**
(latitudes where $Y = 0$ everywhere in longitude) and $2m$ **nodal longitude lines**
(longitude arcs where $Y = 0$ everywhere in latitude).

Confirm this pattern for two harmonics from your plots above.
Then explain physically: what does a harmonic with $m = 0$ look like,
and what does one with $m = n$ look like?  What geophysical signals
might each type represent?

**Your response:**

> *(Write 4–6 sentences.)*


---
## Part 3: Synthesis and the Mauersberger-Lowes Spectrum *(Open — ~40 min)*

You are given a subset of IGRF-14 Gauss coefficients (truncated to degree 8).
Your task is to synthesize the radial component of the magnetic field on a
global grid and compute the Mauersberger-Lowes power spectrum.

**Available hints:** `p3_synthesis`, `p3_spectrum`


In [ ]:
# [PROVIDED] ──────────────────────────────────────────────────────────────────
# IGRF-14 Gauss coefficients (g_n^m, h_n^m) truncated to degree 8, epoch 2025.0.
# Units: nT.  Source: https://www.ngdc.noaa.gov/IAGA/vmod/igrf.html
# Format: (n, m, g_nm, h_nm)  -- h_n^0 = 0 by definition

igrf_coeffs = [
    (1, 0, -29351.0,      0.0),
    (1, 1,  -1410.9,   4545.4),
    (2, 0,  -2555.5,      0.0),
    (2, 1,   2951.4,  -3133.2),
    (2, 2,   1649.6,   -801.1),
    (3, 0,   1316.4,      0.0),
    (3, 1,  -2200.1,   -196.6),
    (3, 2,   1294.3,    250.3),
    (3, 3,    651.5,   -524.6),
    (4, 0,    899.4,      0.0),
    (4, 1,    819.5,    283.0),
    (4, 2,    104.7,   -297.2),
    (4, 3,   -380.9,    -24.0),
    (4, 4,    -19.8,    261.4),
    (5, 0,   -234.9,      0.0),
    (5, 1,    357.5,     15.5),
    (5, 2,    247.9,   -155.1),
    (5, 3,   -153.8,    -66.0),
    (5, 4,    -22.4,    107.0),
    (5, 5,    -81.2,   -178.3),
    (6, 0,     68.6,      0.0),
    (6, 1,     76.1,   -142.3),
    (6, 2,    -54.7,    -31.6),
    (6, 3,   -173.1,    -16.7),
    (6, 4,     -5.9,     70.4),
    (6, 5,     -5.9,     68.0),
    (6, 6,     41.7,    -38.6),
    (7, 0,     24.4,      0.0),
    (7, 1,    142.4,    -72.0),
    (7, 2,    -32.5,    -14.5),
    (7, 3,    -39.3,     -5.0),
    (7, 4,    -57.0,     14.0),
    (7, 5,     11.9,    -62.3),
    (7, 6,     29.8,     56.6),
    (7, 7,     -8.7,    -37.5),
    (8, 0,     24.4,      0.0),
    (8, 1,      8.8,     20.7),
    (8, 2,     22.1,     10.0),
    (8, 3,    -22.2,      0.2),
    (8, 4,    -12.7,     10.7),
    (8, 5,    -16.3,    -11.0),
    (8, 6,    -11.4,    -11.7),
    (8, 7,     10.3,    -10.2),
    (8, 8,     -4.8,     12.2),
]

N_max = 8      # maximum degree
a = 6.371e6    # reference radius (Earth's mean radius), m
print(f'Loaded {len(igrf_coeffs)} IGRF-14 coefficients up to degree {N_max}.')

### Task 3.1 — Synthesize the radial field $B_r$

The radial component of the main field at the surface ($r = a$) from a
potential written in Gauss coefficients is:

$$
B_r(\theta,\phi) = -\frac{\partial V}{\partial r}\bigg|_{r=a}
  = \sum_{n=1}^{N}(n+1)\sum_{m=0}^{n}\bar{P}_n^m(\cos\theta)
    \left[g_n^m\cos(m\phi) + h_n^m\sin(m\phi)\right]
$$

Implement this synthesis and plot $B_r$ as a global map.

Requirements:
- Grid: 1° × 1° in latitude and longitude
- Map projection: simple cylindrical (latitude vs. longitude)
- Diverging colormap, units of nT
- Contours every 5,000 nT overlaid
- Your map should qualitatively resemble the IGRF radial field: roughly dipolar,
  with field entering near the geographic north pole (~−60,000 nT) and
  leaving near the south pole (~+60,000 nT)


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Synthesize Br and plot as a global map.

lat_deg = np.arange(-90, 91, 1.0)
lon_deg = np.arange(0, 361, 1.0)
LAT_G, LON_G = np.meshgrid(lat_deg, lon_deg, indexing='ij')  # shape (181, 361)
theta = np.radians(90.0 - LAT_G)   # colatitude
phi   = np.radians(LON_G)          # longitude

# YOUR CODE HERE

In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Compute and plot the Mauersberger-Lowes spectrum R_n vs. degree n.
#
# R_n = (n+1) * [ g_n0^2 + 2*sum_{m=1}^{n} (g_nm^2 + h_nm^2) ]
#
# Plot on a semi-log scale (log R_n vs. n).
# Also plot a straight line fit to degrees 1–5 and 6–8 separately.
# What do the two slopes tell you about the source of the field?

# YOUR CODE HERE

### Question 3.1 — Interpreting the spectrum

On the Mauersberger-Lowes spectrum, degrees 1–13 typically follow one slope
and degrees above ~14 follow a steeper, roughly flat spectrum.
What is the physical significance of the change in slope?
In your truncated expansion (degrees 1–8) can you already see evidence of
two different source populations?

**Your response:**

> *(Write 4–6 sentences. Reference source depth and the relationship between\ndegree $n$ and spatial wavelength.)*


### Question 3.2 — Truncation and aliasing

Your synthesis used only degrees 1–8.  What spatial features of the global
magnetic field are missing from your map?  Would you expect the errors to be
largest at mid-latitudes, polar regions, or near magnetic anomaly belts?  Why?

**Your response:**

> *(Write 3–5 sentences.)*


---
## Synthesis

### S1 — Connection to Laplace's equation

The spherical harmonic expansion is the general solution to Laplace's equation
$\nabla^2 V = 0$ in spherical coordinates.  Explain in physical terms why only
the decaying terms $(a/r)^{n+1}$ appear in the external potential.
What would a growing term $(r/a)^n$ represent physically, and why is it
excluded for the geomagnetic field observed at Earth's surface?

**Your response:**

> *(Write 4–6 sentences.)*

### S2 — Spherical harmonics vs. Fourier series

Both Fourier series (on a line or plane) and spherical harmonics (on a sphere)
are complete orthogonal bases for representing potential fields.  What are the
advantages of using spherical harmonics for a global field like the IGRF instead
of a global Fourier expansion?  What is the drawback?

**Your response:**

> *(Write 3–5 sentences.)*

### S3 — Upward continuation in spectral domain

The potential at radius $r > a$ can be obtained from the surface potential
by multiplying each degree-$n$ term by $(a/r)^{n+1}$.  This is the spherical
harmonic analog of the Fourier-domain upward continuation filter (covered in
Lab 6).  For which degrees is the continuation factor largest?  What does this
imply about which features survive at satellite altitude?

**Your response:**

> *(Write 4–5 sentences.)*


---
## Extensions *(optional)*

### E1 — Satellite altitude

Compute $B_r$ at satellite altitude (400 km above Earth's surface) by applying
the upward continuation factor $(a/r)^{n+2}$ to each degree.  Plot both the
surface and satellite-altitude maps and compute the ratio of RMS field at each
altitude as a function of degree $n$.

### E2 — Secular variation

IGRF also provides secular variation coefficients $\dot{g}_n^m$ (rate of change
in nT/yr).  Predict the field at epoch 2030 by advancing the 2025 coefficients
by 5 years.  Where on the globe does $B_r$ change most rapidly?

### E3 — Spectral leakage

Compute the spatial resolution (approximate minimum wavelength in km) for each
degree $n$ of the expansion and create a table.  At what degree would you need
to truncate to resolve a 500 km × 500 km anomaly?  Compare with the achievable
truncation from ground-based vs. satellite measurements.


In [ ]:
# [EXPLORE] ───────────────────────────────────────────────────────────────────
# Extension workspace.

# YOUR CODE HERE